<a href="https://colab.research.google.com/github/sarshadad-codeee/FlyRank_ML_Task1/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/sarshadad-codeee/FlyRank_ML_Task1"
REPO_DIR = "FlyRank_ML_Task1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working directory:", os.getcwd())
!pip install duckdb --quiet

Working directory: /content/FlyRank_ML_Task1


In [4]:
import duckdb
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

base = "hf://datasets/FlyRank/internship-warehouse"

In [3]:
march_path = f"{base}/fact_content_daily_performance/month=2026-03/*.parquet"
april_path = f"{base}/fact_content_daily_performance/month=2026-04/*.parquet"

march_df = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_clicks) AS march_clicks,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_sum_position) AS march_sum_position
    FROM read_parquet('{march_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

march_df["avg_position"] = march_df["march_sum_position"] / march_df["march_impressions"]
march_df["ctr"] = march_df["march_clicks"] / march_df["march_impressions"].replace(0, pd.NA)

april_df = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_clicks) AS april_clicks
    FROM read_parquet('{april_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

merged = march_df.merge(april_df, on=["content_hash_id", "client_hash_id"], how="inner")
merged["is_declining"] = (merged["april_clicks"] < merged["march_clicks"]).astype(int)

print(f"Merged shape: {merged.shape}")
print(f"Base rate (declining): {merged['is_declining'].mean():.3f}")
merged.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Merged shape: (158549, 9)
Base rate (declining): 0.279


,content_hash_id,client_hash_id,march_clicks,march_impressions,march_sum_position,avg_position,ctr,april_clicks,is_declining
0,content_05597932fe4da067,client_73cda7b4e4f265ea,0.0,57.0,131.0,2.298246,0.000000,0.0,0
1,content_7a105f548d9c6916,client_73cda7b4e4f265ea,7.0,6523.0,44965.0,6.893301,0.001073,8.0,0
2,content_905aa32a0230694e,client_73cda7b4e4f265ea,0.0,149.0,840.0,5.637584,0.000000,0.0,0
3,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,0.0,453.0,1456.0,3.214128,0.000000,2.0,0
4,content_36c36abc7650d7af,client_73cda7b4e4f265ea,6.0,5630.0,36794.0,6.535346,0.001066,4.0,1


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [5]:
"""
Method choice: Logistic Regression, then Random Forest (per the
training-honest-models skill's method table: "yes/no with an observed
label -> Logistic Regression, then Random Forest, because readable ->
stronger").

My question is a "which first?" ranking problem (per ML-03's framing:
prioritizing content for CTR-fix review), evaluated with precision@50
-- so I use each model's predicted PROBABILITY of decline as the
ranking score, not a hard yes/no label. This matches the skill's
guidance: "ranking needs scores, not labels."

I start with Logistic Regression for its readability -- I can inspect
its coefficients directly and sanity-check whether they make sense
before trusting a stronger, less transparent model. Random Forest is
added second, only if it meaningfully beats the simpler model on the
same metric -- per the skill's rule, "simplicity is a feature... add
complexity only when the comparison earns it."

Label: is_declining (April clicks < March clicks for the same
content+client) -- built using genuine FORWARD data (April), not a
same-period trick. This avoids the exact leakage trap caught in ML-04:
the label is never built from the same period as the features.
"""

'\nMethod choice: Logistic Regression, then Random Forest (per the \ntraining-honest-models skill\'s method table: "yes/no with an observed \nlabel -> Logistic Regression, then Random Forest, because readable -> \nstronger").\n\nMy question is a "which first?" ranking problem (per ML-03\'s framing: \nprioritizing content for CTR-fix review), evaluated with precision@50 \n-- so I use each model\'s predicted PROBABILITY of decline as the \nranking score, not a hard yes/no label. This matches the skill\'s \nguidance: "ranking needs scores, not labels."\n\nI start with Logistic Regression for its readability -- I can inspect \nits coefficients directly and sanity-check whether they make sense \nbefore trusting a stronger, less transparent model. Random Forest is \nadded second, only if it meaningfully beats the simpler model on the \nsame metric -- per the skill\'s rule, "simplicity is a feature... add \ncomplexity only when the comparison earns it."\n\nLabel: is_declining (April clicks < 

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [8]:
"""
Split: grouped by client_hash_id, NOT a random row-level split.

This is the same principle established in ML-02 and ML-04: the same
client's content must never appear in both train and test, or the
model could learn client-specific quirks (e.g. one client's unusually
good/bad content patterns) rather than genuinely general signal --
producing an optimistic, dishonest score.

Not time-aware in the traditional sense (train on earlier months, test
on later ones) because both March (features) and April (label) are
already used together to define each row -- the forward-looking
element is already built into the label itself, not into the split. A
future capstone iteration could additionally hold out an entirely
separate month for true walk-forward validation.
"""

"\nSplit: grouped by client_hash_id, NOT a random row-level split.\n\nThis is the same principle established in ML-02 and ML-04: the same \nclient's content must never appear in both train and test, or the \nmodel could learn client-specific quirks (e.g. one client's unusually \ngood/bad content patterns) rather than genuinely general signal -- \nproducing an optimistic, dishonest score.\n\nNot time-aware in the traditional sense (train on earlier months, test \non later ones) because both March (features) and April (label) are \nalready used together to define each row -- the forward-looking \nelement is already built into the label itself, not into the split. A \nfuture capstone iteration could additionally hold out an entirely \nseparate month for true walk-forward validation.\n"

In [9]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols = ["march_clicks", "march_impressions", "avg_position", "ctr"]
X = merged[feature_cols].fillna(0)
y = merged["is_declining"]
groups = merged["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Confirm no client overlap
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
print(f"Train rows: {len(X_train)}, Test rows: {len(X_test)}")
print(f"Client overlap between train/test: {len(train_clients & test_clients)}")
print(f"Base rate train: {y_train.mean():.3f}, Base rate test: {y_test.mean():.3f}")

Train rows: 107358, Test rows: 51191
Client overlap between train/test: 0
Base rate train: 0.255, Base rate test: 0.331


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [10]:
"""
Comparison design: baseline rule (from ML-07) and both models are
scored on the SAME test set, using the SAME metric (precision@50),
in this same notebook run -- per the training-honest-models skill's
non-negotiable rule.

The ML-07 baseline rule (visible_but_weak_position: impressions >= 100
AND avg_position >= 20, scored by impressions) was designed for a
same-period signal, not this notebook's forward-looking label -- so
here it is re-applied to the same test set and evaluated honestly
against the real forward outcome (is_declining), not assumed to still
work well just because it worked as a rule before.
"""

"\nComparison design: baseline rule (from ML-07) and both models are \nscored on the SAME test set, using the SAME metric (precision@50), \nin this same notebook run -- per the training-honest-models skill's \nnon-negotiable rule.\n\nThe ML-07 baseline rule (visible_but_weak_position: impressions >= 100 \nAND avg_position >= 20, scored by impressions) was designed for a \nsame-period signal, not this notebook's forward-looking label -- so \nhere it is re-applied to the same test set and evaluated honestly \nagainst the real forward outcome (is_declining), not assumed to still \nwork well just because it worked as a rule before.\n"

In [11]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# --- Baseline (ML-07 rule, re-applied to this test set) ---
test_df = merged.iloc[test_idx].copy()
has_volume = (test_df["march_impressions"] >= 100).astype(int)
weak_position = (test_df["avg_position"] >= 20).astype(int)
baseline_score = has_volume * weak_position * test_df["march_impressions"]

# --- Logistic Regression ---
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train, y_train)
logreg_scores = logreg.predict_proba(X_test)[:, 1]

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

# --- Comparison table ---
base_rate = y_test.mean()
results = {
    "Baseline (ML-07 rule)": precision_at_k(baseline_score, y_test.values, 50),
    "Logistic Regression":   precision_at_k(logreg_scores, y_test.values, 50),
    "Random Forest":         precision_at_k(rf_scores, y_test.values, 50),
}

print(f"Base rate (random guessing baseline): {base_rate:.3f}\n")
for name, score in results.items():
    print(f"{name:25s} Precision@50 = {score:.3f}")

Base rate (random guessing baseline): 0.331

Baseline (ML-07 rule)     Precision@50 = 0.640
Logistic Regression       Precision@50 = 0.660
Random Forest             Precision@50 = 0.780


In [12]:
"""
Results: Random Forest clearly wins at Precision@50 (0.780), beating
both the ML-07 baseline rule (0.640) and Logistic Regression (0.660).
The base rate (0.331) confirms this isn't a trivially easy target --
random guessing would only get roughly a third right, so the baseline
rule's 0.640 was already meaningfully better than chance, and Random
Forest adds a further real, non-trivial improvement on top of that.

Logistic Regression's marginal gain over the rule (+0.02) suggests the
relationship between these features and decline isn't strongly linear
-- Random Forest's ability to capture feature interactions and
non-linear thresholds is likely what drives its stronger result. This
will be checked against feature importance next, to confirm the model
isn't relying on something suspicious.
"""

"\nResults: Random Forest clearly wins at Precision@50 (0.780), beating \nboth the ML-07 baseline rule (0.640) and Logistic Regression (0.660). \nThe base rate (0.331) confirms this isn't a trivially easy target -- \nrandom guessing would only get roughly a third right, so the baseline \nrule's 0.640 was already meaningfully better than chance, and Random \nForest adds a further real, non-trivial improvement on top of that.\n\nLogistic Regression's marginal gain over the rule (+0.02) suggests the \nrelationship between these features and decline isn't strongly linear \n-- Random Forest's ability to capture feature interactions and \nnon-linear thresholds is likely what drives its stronger result. This \nwill be checked against feature importance next, to confirm the model \nisn't relying on something suspicious.\n"

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [13]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Random Forest feature importances:")
print(importances)

Random Forest feature importances:
ctr                  0.482474
march_clicks         0.427023
march_impressions    0.069438
avg_position         0.021065
dtype: float64


In [14]:
# Attach RF predictions back to the test set for error inspection
test_df["rf_score"] = rf_scores
test_df["actual"] = y_test.values
test_df["predicted_declining"] = (rf_scores >= 0.5).astype(int)

false_positives = test_df[(test_df["predicted_declining"] == 1) & (test_df["actual"] == 0)]
false_negatives = test_df[(test_df["predicted_declining"] == 0) & (test_df["actual"] == 1)]

print(f"False positives (predicted decline, actually fine): {len(false_positives)}")
print(f"False negatives (missed a real decline): {len(false_negatives)}\n")

print("Sample false positives:")
print(false_positives[["content_hash_id", "march_clicks", "march_impressions", "avg_position", "ctr", "rf_score"]].head(3))

print("\nSample false negatives:")
print(false_negatives[["content_hash_id", "march_clicks", "march_impressions", "avg_position", "ctr", "rf_score"]].head(3))

False positives (predicted decline, actually fine): 8996
False negatives (missed a real decline): 231

Sample false positives:
            content_hash_id  march_clicks  march_impressions  avg_position  \
1  content_7a105f548d9c6916           7.0             6523.0      6.893301   
5  content_05434271b257bb68           6.0             1421.0      6.906404   
8  content_5d412fba6e1a2582           1.0              223.0     10.538117   

        ctr  rf_score  
1  0.001073  0.641082  
5  0.004222  0.656947  
8  0.004484  0.682981  

Sample false negatives:
              content_hash_id  march_clicks  march_impressions  avg_position  \
273  content_2cd9d1a4563c54ae           1.0             5677.0      3.011097   
282  content_8e1334d6356668e3           1.0           134984.0      2.693038   
283  content_9bcfb1e373c01b7a           1.0             9370.0      4.333511   

          ctr  rf_score  
273  0.000176  0.456448  
282  0.000007  0.235608  
283  0.000107  0.434612  


In [16]:
"""
Feature importance: ctr (0.482) and march_clicks (0.427) together
account for ~91% of the model's decisions, with march_impressions
(0.069) and avg_position (0.021) contributing much less. Neither
feature is "suspiciously perfect" (no single feature dominates at
0.90+), so this doesn't look like the kind of leakage caught in ML-04
-- but it's worth naming honestly what this likely reflects: content
with more March clicks has more room to decline in April (regression
toward the mean), so the model may be partly learning "high performers
are more likely to fall" rather than a deeper causal driver of decline.
This is a legitimate, real pattern -- not a flaw -- but it means the
model's strength may not transfer as cleanly to lower-traffic content,
where there's less room to fall in absolute terms.

Errors, 3 concrete cases:

False positive (content_7a105f548d9c6916): march_clicks=7, ctr=0.001,
avg_position=6.9, rf_score=0.64. Model flagged this as likely to
decline based on its low CTR relative to a decent position -- but it
didn't. This is a genuinely hard case: the signals look weak by the
model's logic, but April's actual outcome didn't follow the pattern,
suggesting some months are just noisy at this volume level.

False negative (content_8e1334d6356668e3): march_clicks=1,
impressions=134,984, ctr=0.000007 (near zero), rf_score=0.24 (missed).
This is the hardest case in the set: the content already has enormous
impressions but almost no clicks -- it's already near rock-bottom
performance, so there was little room left to "decline" in a way the
model's current features could detect. This suggests the model
struggles with already-severely-underperforming content, since
march_clicks (a top feature) offers almost no signal when it's already
close to zero.

False negative (content_2cd9d1a4563c54ae): march_clicks=1,
impressions=5,677, ctr=0.0002, rf_score=0.46 (just below the 0.5
threshold, close miss). Similar pattern to the case above -- very low
starting click count leaves little room for the model to distinguish
"about to decline further" from "already at its floor."

Overall: the model's errors cluster around already-low-performing
content, where march_clicks (the model's second most important
feature) carries little information because it's already near its
floor. A useful next iteration would add a feature capturing relative
change over a longer trailing window, not just the single prior month,
to better separate "stable at a low level" from "actively declining."
"""

'\nFeature importance: ctr (0.482) and march_clicks (0.427) together \naccount for ~91% of the model\'s decisions, with march_impressions \n(0.069) and avg_position (0.021) contributing much less. Neither \nfeature is "suspiciously perfect" (no single feature dominates at \n0.90+), so this doesn\'t look like the kind of leakage caught in ML-04 \n-- but it\'s worth naming honestly what this likely reflects: content \nwith more March clicks has more room to decline in April (regression \ntoward the mean), so the model may be partly learning "high performers \nare more likely to fall" rather than a deeper causal driver of decline. \nThis is a legitimate, real pattern -- not a flaw -- but it means the \nmodel\'s strength may not transfer as cleanly to lower-traffic content, \nwhere there\'s less room to fall in absolute terms.\n\nErrors, 3 concrete cases:\n\nFalse positive (content_7a105f548d9c6916): march_clicks=7, ctr=0.001, \navg_position=6.9, rf_score=0.64. Model flagged this as likely

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.